<a href="https://www.kaggle.com/code/dhruvjain35/triagegeist-clinical-ai-pipeline?scriptVersionId=306773248" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
print("""
DATASET CITATION AND DISCLOSURE
Dataset: Triagegeist Synthetic ED Dataset
Source: Laitinen-Fredriksson Foundation
Access: kaggle.com/competitions/triagegeist/data
License: Non-Commercial Research License
Files used: train.csv, test.csv, chief_complaints.csv, patient_history.csv
No external datasets were used.
""")

In [ ]:
# loading everything in and merging the four files on patient_id
# path is /kaggle/input/competitions/triagegeist/ for this competition
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import cohen_kappa_score, accuracy_score, confusion_matrix
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
import shap
import warnings
warnings.filterwarnings('ignore')

PATH = '/kaggle/input/competitions/triagegeist/'
train = pd.read_csv(PATH + 'train.csv')
test = pd.read_csv(PATH + 'test.csv')
complaints = pd.read_csv(PATH + 'chief_complaints.csv')
history = pd.read_csv(PATH + 'patient_history.csv')

train = train.merge(complaints, on='patient_id', how='left')
train = train.merge(history, on='patient_id', how='left')
test = test.merge(complaints, on='patient_id', how='left')
test = test.merge(history, on='patient_id', how='left')

print(f"Train: {train.shape} | Test: {test.shape}")
print("""
DATASET CITATION AND DISCLOSURE
Dataset: Triagegeist Synthetic ED Dataset
Source: Laitinen-Fredriksson Foundation
Access: kaggle.com/competitions/triagegeist/data
License: Non-Commercial Research License
Files used: train.csv, test.csv, chief_complaints.csv, patient_history.csv
No external datasets were used.
""")

In [ ]:
# this is the main feature engineering function
# key design choices:
# - missingness in BP/RR is clinically informative so i made indicator columns before imputing
# - binary clinical flags (tachycardia, hypoxia etc) based on standard ED thresholds
# - high-risk keyword flags capture critical clinical phrases tfidf alone misses
# - tfidf on chief complaints with trigrams for richer phrase detection
# - disposition and ed_los_hours are dropped here — both are post-triage leakage

LEAKAGE = ['disposition', 'ed_los_hours']
IDS = ['patient_id']
CAT_COLS = ['sex', 'insurance_type', 'language', 'age_group', 'arrival_mode',
            'mental_status_triage', 'arrival_day', 'arrival_season', 'shift',
            'transport_origin', 'pain_location', 'chief_complaint_system',
            'site_id', 'triage_nurse_id']

HIGH_RISK_PATTERNS = {
    'kw_chest_pain':    r'chest pain|chest tightness|chest pressure',
    'kw_resp_distress': r'shortness of breath|difficulty breathing|cant breathe',
    'kw_neuro':         r'altered mental|confusion|unresponsive|unconscious',
    'kw_stroke':        r'stroke|facial droop|arm weakness|sudden weakness',
    'kw_seizure':       r'seizure|convulsion|postictal',
    'kw_trauma':        r'trauma|motor vehicle|mva|fall from height|assault',
    'kw_overdose':      r'overdose|ingestion|poisoning|intoxication',
    'kw_severe_pain':   r'10 out of 10|severe pain|excruciating|worst pain',
    'kw_syncope':       r'syncope|fainted|passed out|loss of consciousness',
    'kw_thunderclap':   r'thunderclap|worst headache|sudden severe headache',
    'kw_bleeding':      r'hemorrhage|severe bleeding|hemoptysis|hematemesis',
    'kw_sepsis':        r'sepsis|septic|high fever chills|bacteremia',
    'kw_cardiac':       r'heart attack|myocardial|cardiac arrest|palpitations',
    'kw_anaphylaxis':   r'anaphylaxis|allergic reaction|throat swelling',
    'kw_psychiatric':   r'suicidal|homicidal|psychosis|self harm',
}

def prepare_data(df, fit=True, artifacts=None):
    d = df.copy()
    d = d.drop(columns=[c for c in LEAKAGE + IDS + ['chief_complaint_system_y'] if c in d.columns])
    if 'chief_complaint_system_x' in d.columns:
        d = d.rename(columns={'chief_complaint_system_x': 'chief_complaint_system'})

    d['pain_score_missing'] = (d['pain_score'] == -1).astype(int)
    d['pain_score'] = d['pain_score'].replace(-1, np.nan)

    for col in ['systolic_bp', 'diastolic_bp', 'mean_arterial_pressure',
                'pulse_pressure', 'shock_index', 'respiratory_rate']:
        d[f'{col}_missing'] = d[col].isnull().astype(int)

    d['fever'] = (d['temperature_c'] >= 38.0).astype(int)
    d['hypothermia'] = (d['temperature_c'] < 36.0).astype(int)
    d['temp_deviation'] = (d['temperature_c'] - 37.0).abs()
    d['tachycardia'] = (d['heart_rate'] > 100).astype(int)
    d['bradycardia'] = (d['heart_rate'] < 60).astype(int)
    d['hypoxia'] = (d['spo2'] < 94).astype(int)
    d['severe_hypoxia'] = (d['spo2'] < 88).astype(int)
    d['critical_hypoxia'] = (d['spo2'] < 85).astype(int)
    d['hypotension'] = ((d['systolic_bp'].notna()) & (d['systolic_bp'] < 90)).astype(int)
    d['severe_hypotension'] = ((d['systolic_bp'].notna()) & (d['systolic_bp'] < 70)).astype(int)
    d['hypertensive_crisis'] = ((d['systolic_bp'].notna()) & (d['systolic_bp'] >= 180)).astype(int)
    d['tachypnea'] = ((d['respiratory_rate'].notna()) & (d['respiratory_rate'] > 20)).astype(int)
    d['severe_tachypnea'] = ((d['respiratory_rate'].notna()) & (d['respiratory_rate'] > 30)).astype(int)
    d['bradypnea'] = ((d['respiratory_rate'].notna()) & (d['respiratory_rate'] < 10)).astype(int)
    d['shock_index_elevated'] = ((d['shock_index'].notna()) & (d['shock_index'] > 1.0)).astype(int)
    d['shock_index_critical'] = ((d['shock_index'].notna()) & (d['shock_index'] > 1.4)).astype(int)
    d['critical_gcs'] = (d['gcs_total'] <= 8).astype(int)
    d['moderate_gcs'] = ((d['gcs_total'] > 8) & (d['gcs_total'] < 13)).astype(int)
    d['altered_mental'] = d['mental_status_triage'].isin(
        ['confused', 'drowsy', 'agitated', 'unresponsive']
    ).astype(int)
    d['is_elderly'] = (d['age'] >= 65).astype(int)
    d['is_very_elderly'] = (d['age'] >= 85).astype(int)
    d['is_pediatric'] = (d['age'] < 18).astype(int)
    d['is_infant'] = (d['age'] < 2).astype(int)
    d['age_gcs'] = d['age'] * d['gcs_total']
    d['overnight_arrival'] = ((d['arrival_hour'] >= 22) | (d['arrival_hour'] <= 6)).astype(int)
    d['weekend'] = d['arrival_day'].isin(['Saturday', 'Sunday']).astype(int)
    d['hour_sin'] = np.sin(2 * np.pi * d['arrival_hour'] / 24)
    d['hour_cos'] = np.cos(2 * np.pi * d['arrival_hour'] / 24)

    hx_cols = [c for c in d.columns if c.startswith('hx_')]
    d['comorbidity_burden'] = d[hx_cols].sum(axis=1)
    d['high_comorbidity'] = (d['comorbidity_burden'] >= 3).astype(int)
    d['multi_comorbidity'] = (d['comorbidity_burden'] >= 5).astype(int)
    high_risk_hx = ['hx_heart_failure', 'hx_malignancy', 'hx_copd', 'hx_ckd',
                    'hx_coronary_artery_disease', 'hx_coagulopathy', 'hx_immunosuppressed']
    d['high_risk_comorbidity'] = d[[c for c in high_risk_hx if c in d.columns]].max(axis=1)

    complaint_text = d['chief_complaint_raw'].fillna('unknown').str.lower()
    for col, pattern in HIGH_RISK_PATTERNS.items():
        d[col] = complaint_text.str.contains(pattern, regex=True, na=False).astype(int)
    d['kw_total'] = d[[k for k in HIGH_RISK_PATTERNS.keys()]].sum(axis=1)

    impute_cols = ['systolic_bp', 'diastolic_bp', 'heart_rate', 'respiratory_rate',
                   'temperature_c', 'spo2', 'mean_arterial_pressure', 'pulse_pressure',
                   'shock_index', 'pain_score']

    if fit:
        artifacts = {}
        artifacts['medians'] = {col: d[col].median() for col in impute_cols if col in d.columns}

    for col in impute_cols:
        if col in d.columns:
            d[col] = d[col].fillna(artifacts['medians'][col])

    if fit:
        artifacts['encoders'] = {}
        for col in CAT_COLS:
            if col in d.columns:
                le = LabelEncoder()
                d[col] = le.fit_transform(d[col].astype(str))
                artifacts['encoders'][col] = le
    else:
        for col in CAT_COLS:
            if col in d.columns and col in artifacts['encoders']:
                le = artifacts['encoders'][col]
                d[col] = d[col].astype(str).apply(
                    lambda x: le.transform([x])[0] if x in le.classes_ else -1
                )

    cc = d['chief_complaint_raw'].fillna('unknown')
    d = d.drop(columns=['chief_complaint_raw'])

    if fit:
        tfidf = TfidfVectorizer(max_features=200, ngram_range=(1, 3), min_df=3, sublinear_tf=True)
        tfidf_mat = tfidf.fit_transform(cc)
        artifacts['tfidf'] = tfidf
    else:
        tfidf_mat = artifacts['tfidf'].transform(cc)

    tfidf_df = pd.DataFrame(
        tfidf_mat.toarray(),
        columns=[f'cc_{i}' for i in range(tfidf_mat.shape[1])],
        index=d.index
    )
    d = pd.concat([d, tfidf_df], axis=1)
    return d, artifacts

def qwk(y_true, y_pred):
    return cohen_kappa_score(y_true, y_pred, weights='quadratic')

def lgb_qwk_eval(y_pred, dataset):
    y_true = dataset.get_label().astype(int)
    y_pred_class = np.argmax(y_pred.reshape(len(y_true), 5), axis=1)
    return 'qwk', cohen_kappa_score(y_true, y_pred_class, weights='quadratic'), True

TARGET = 'triage_acuity'
N_FOLDS = 5
y = train[TARGET].values
X_raw = train.drop(columns=[TARGET])

X_full, artifacts = prepare_data(X_raw, fit=True)
X_test, _ = prepare_data(test.copy(), fit=False, artifacts=artifacts)

X_main, X_cal, y_main, y_cal = train_test_split(
    X_full, y, test_size=0.10, random_state=42, stratify=y
)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

print(f"Train: {X_main.shape} | Calibration: {X_cal.shape} | Test: {X_test.shape}")
print(f"Total features: {X_full.shape[1]}")
print(f"Keyword features added: {len(HIGH_RISK_PATTERNS) + 1}")
print("\nLEAKAGE AUDIT")
print("Excluded: disposition, ed_los_hours (post-triage outcomes)")
print("No test set information used during training or calibration.")

In [ ]:
# 5-fold stratified CV with lightgbm
# using QWK as eval metric because this is an ordinal problem — wrong by 3 levels
# should be penalized more than wrong by 1
# early stopping at 100 rounds per fold to avoid overfitting

lgb_params = {
    'objective': 'multiclass',
    'num_class': 5,
    'metric': 'multi_logloss',
    'learning_rate': 0.05,
    'num_leaves': 127,
    'min_child_samples': 20,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'n_jobs': -1,
    'seed': 42,
    'verbose': -1,
}

oof_lgb_probs = np.zeros((len(X_main), 5))
test_lgb_probs = np.zeros((len(X_test), 5))
lgb_models = []
lgb_fold_qwks = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_main, y_main)):
    X_tr, y_tr = X_main.iloc[tr_idx], y_main[tr_idx] - 1
    X_val, y_val = X_main.iloc[val_idx], y_main[val_idx] - 1

    dtrain = lgb.Dataset(X_tr, label=y_tr)
    dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)

    model = lgb.train(
        lgb_params, dtrain,
        num_boost_round=2000,
        valid_sets=[dval],
        feval=lgb_qwk_eval,
        callbacks=[
            lgb.early_stopping(100, verbose=False),
            lgb.log_evaluation(200)
        ]
    )

    val_probs = model.predict(X_val, num_iteration=model.best_iteration)
    val_preds = np.argmax(val_probs, axis=1) + 1
    fqwk = qwk(y_main[val_idx], val_preds)
    lgb_fold_qwks.append(fqwk)
    oof_lgb_probs[val_idx] = val_probs
    test_lgb_probs += model.predict(X_test, num_iteration=model.best_iteration) / N_FOLDS
    lgb_models.append(model)
    print(f"LGB Fold {fold+1} | Iter: {model.best_iteration} | QWK: {fqwk:.4f}")

lgb_oof_preds = np.argmax(oof_lgb_probs, axis=1) + 1
lgb_oof_qwk = qwk(y_main, lgb_oof_preds)
print(f"\nLightGBM OOF QWK: {lgb_oof_qwk:.4f}")

In [ ]:
# 5-fold stratified CV with xgboost
# depth-wise tree growth complements lightgbm's leaf-wise approach
# the two models together reduce variance through diversity

xgb_params = {
    'objective': 'multi:softprob',
    'num_class': 5,
    'eval_metric': 'mlogloss',
    'learning_rate': 0.05,
    'max_depth': 7,
    'min_child_weight': 5,
    'subsample': 0.80,
    'colsample_bytree': 0.65,
    'reg_alpha': 0.05,
    'reg_lambda': 1.0,
    'seed': 42,
    'nthread': -1,
    'verbosity': 0,
    'tree_method': 'hist',
}

oof_xgb_probs = np.zeros((len(X_main), 5))
test_xgb_probs = np.zeros((len(X_test), 5))
xgb_models = []
xgb_fold_qwks = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_main, y_main)):
    X_tr, y_tr = X_main.iloc[tr_idx], y_main[tr_idx] - 1
    X_val, y_val = X_main.iloc[val_idx], y_main[val_idx] - 1

    dm_tr = xgb.DMatrix(X_tr, label=y_tr)
    dm_val = xgb.DMatrix(X_val, label=y_val)
    dm_te = xgb.DMatrix(X_test)

    xgb_model = xgb.train(
        xgb_params, dm_tr,
        num_boost_round=2000,
        evals=[(dm_val, 'val')],
        early_stopping_rounds=100,
        verbose_eval=False,
    )

    val_probs = xgb_model.predict(dm_val).reshape(-1, 5)
    val_preds = np.argmax(val_probs, axis=1) + 1
    fqwk = qwk(y_main[val_idx], val_preds)
    xgb_fold_qwks.append(fqwk)
    oof_xgb_probs[val_idx] = val_probs
    test_xgb_probs += xgb_model.predict(dm_te).reshape(-1, 5) / N_FOLDS
    xgb_models.append(xgb_model)
    print(f"XGB Fold {fold+1} | Iter: {xgb_model.best_iteration} | QWK: {fqwk:.4f}")

xgb_oof_preds = np.argmax(oof_xgb_probs, axis=1) + 1
xgb_oof_qwk = qwk(y_main, xgb_oof_preds)
print(f"\nXGBoost OOF QWK: {xgb_oof_qwk:.4f}")

In [ ]:
# 5-fold stratified CV with catboost
# ordered target statistics handle categories differently to lgb and xgb
# three diverse models together cover more of the hypothesis space than any single model

cat_params = {
    'iterations': 1000,
    'learning_rate': 0.05,
    'depth': 7,
    'l2_leaf_reg': 3.0,
    'loss_function': 'MultiClass',
    'eval_metric': 'Accuracy',
    'early_stopping_rounds': 100,
    'random_seed': 42,
    'verbose': 200,
    'thread_count': -1,
    'use_best_model': True,
}

oof_cat_probs = np.zeros((len(X_main), 5))
test_cat_probs = np.zeros((len(X_test), 5))
cat_models = []
cat_fold_qwks = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_main, y_main)):
    X_tr, y_tr = X_main.iloc[tr_idx], y_main[tr_idx] - 1
    X_val, y_val = X_main.iloc[val_idx], y_main[val_idx] - 1

    cat_model = CatBoostClassifier(**cat_params)
    cat_model.fit(X_tr, y_tr, eval_set=(X_val, y_val))

    val_probs = cat_model.predict_proba(X_val)
    val_preds = np.argmax(val_probs, axis=1) + 1
    fqwk = qwk(y_main[val_idx], val_preds)
    cat_fold_qwks.append(fqwk)
    oof_cat_probs[val_idx] = val_probs
    test_cat_probs += cat_model.predict_proba(X_test) / N_FOLDS
    cat_models.append(cat_model)
    print(f"CAT Fold {fold+1} | QWK: {fqwk:.4f}")

cat_oof_preds = np.argmax(oof_cat_probs, axis=1) + 1
cat_oof_qwk = qwk(y_main, cat_oof_preds)
print(f"\nCatBoost OOF QWK: {cat_oof_qwk:.4f}")

In [ ]:
# performance-weighted ensemble blend
# each model weighted proportionally to its OOF QWK score
# better performing models get more say in the final prediction

total_w = lgb_oof_qwk + xgb_oof_qwk + cat_oof_qwk
w_lgb = lgb_oof_qwk / total_w
w_xgb = xgb_oof_qwk / total_w
w_cat = cat_oof_qwk / total_w

print(f"Ensemble weights:")
print(f"  LightGBM: {w_lgb:.3f} (OOF QWK: {lgb_oof_qwk:.4f})")
print(f"  XGBoost:  {w_xgb:.3f} (OOF QWK: {xgb_oof_qwk:.4f})")
print(f"  CatBoost: {w_cat:.3f} (OOF QWK: {cat_oof_qwk:.4f})")

oof_ensemble_probs = w_lgb * oof_lgb_probs + w_xgb * oof_xgb_probs + w_cat * oof_cat_probs
test_ensemble_probs = w_lgb * test_lgb_probs + w_xgb * test_xgb_probs + w_cat * test_cat_probs

oof_preds = np.argmax(oof_ensemble_probs, axis=1) + 1
oof_qwk = qwk(y_main, oof_preds)
oof_acc = accuracy_score(y_main, oof_preds)

print(f"\n{'='*55}")
print(f"ENSEMBLE OOF QWK:      {oof_qwk:.4f}  |  NEWS2 baseline: 0.7723")
print(f"ENSEMBLE OOF Accuracy: {oof_acc:.4f}  |  NEWS2 baseline: 0.4076")
print(f"Delta QWK vs NEWS2:    {oof_qwk - 0.7723:+.4f}")
print(f"\nIndividual vs ensemble:")
print(f"  LightGBM: {lgb_oof_qwk:.4f}")
print(f"  XGBoost:  {xgb_oof_qwk:.4f}")
print(f"  CatBoost: {cat_oof_qwk:.4f}")
print(f"  Ensemble: {oof_qwk:.4f}")

In [ ]:
# checking if the model makes more mistakes for certain demographic groups
# undertriage = predicted a less urgent level than the true label
# critical miss = true ESI 1 or 2 predicted as ESI 3 or higher (the dangerous error)
# chi-squared test to check if differences between groups are statistically significant
oof_df = X_main.copy()
oof_df['true'] = y_main
oof_df['pred'] = oof_preds
oof_df['undertriage'] = (oof_df['pred'] > oof_df['true']).astype(int)
oof_df['critical_true'] = (oof_df['true'] <= 2).astype(int)
oof_df['critical_missed'] = ((oof_df['true'] <= 2) & (oof_df['pred'] > 2)).astype(int)

enc = artifacts['encoders']

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
axes = axes.flatten()

bias_targets = [('sex', 'Sex'), ('insurance_type', 'Insurance Type'),
                ('language', 'Language'), ('age_group', 'Age Group')]

for i, (col, label) in enumerate(bias_targets):
    if col not in enc:
        continue
    le = enc[col]
    decoded = le.inverse_transform(oof_df[col].values)

    stats_df = pd.DataFrame({'group': decoded, 'undertriage': oof_df['undertriage'].values,
                              'critical_missed': oof_df['critical_missed'].values,
                              'critical_true': oof_df['critical_true'].values})

    undertriage_rate = stats_df.groupby('group')['undertriage'].mean().sort_values(ascending=False)
    critical_miss = stats_df[stats_df['critical_true'] == 1].groupby('group')['critical_missed'].mean()

    x = np.arange(len(undertriage_rate))
    width = 0.35
    bars1 = axes[i].bar(x - width/2, undertriage_rate.values * 100, width,
                         label='Undertriage rate', color='#d62728', alpha=0.8)
    
    aligned_critical = critical_miss.reindex(undertriage_rate.index).fillna(0)
    bars2 = axes[i].bar(x + width/2, aligned_critical.values * 100, width,
                         label='Critical miss rate', color='#ff7f0e', alpha=0.8)

    axes[i].axhline(undertriage_rate.mean() * 100, color='gray', linestyle='--',
                     alpha=0.7, label=f'Mean undertriage: {undertriage_rate.mean()*100:.1f}%')
    axes[i].set_xticks(x)
    axes[i].set_xticklabels(undertriage_rate.index, rotation=30, ha='right', fontsize=9)
    axes[i].set_ylabel('Rate (%)', fontsize=10)
    axes[i].set_title(f'Undertriage & Critical Miss Rate by {label}', fontsize=11, fontweight='bold')
    axes[i].legend(fontsize=8)

    print(f"\n{label.upper()} UNDERTRIAGE RATES:")
    for grp in undertriage_rate.index:
        n = (decoded == grp).sum()
        ut = undertriage_rate[grp] * 100
        cm_rate = critical_miss.get(grp, 0) * 100
        print(f"  {grp:20s} n={n:6,}  undertriage={ut:.1f}%  critical_miss={cm_rate:.1f}%")

    chi2, p_val = stats.chi2_contingency(
        pd.crosstab(decoded, oof_df['undertriage'].values)
    )[:2]
    print(f"  Chi-squared test: p={p_val:.4f} ({'SIGNIFICANT' if p_val < 0.05 else 'not significant'})")

plt.suptitle('Systematic Undertriage Audit — Demographic Bias Analysis\n(OOF Predictions vs. True ESI Labels)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('bias_audit.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# shap to explain what the model is actually doing
# running on a 2000 patient sample to keep it fast
# splitting by ESI class so i can see what drives critical (level 1) predictions specifically

best_lgb_model = lgb_models[int(np.argmax(lgb_fold_qwks))]
explainer = shap.TreeExplainer(best_lgb_model)

sample_idx = np.random.choice(len(X_main), size=2000, replace=False)
X_sample = X_main.iloc[sample_idx]
shap_values_raw = explainer.shap_values(X_sample)

if isinstance(shap_values_raw, np.ndarray) and shap_values_raw.ndim == 3:
    shap_values = [shap_values_raw[:, :, i] for i in range(shap_values_raw.shape[2])]
else:
    shap_values = shap_values_raw

mean_shap = np.mean([np.abs(shap_values[i]).mean(axis=0) for i in range(5)], axis=0)
top_features = pd.Series(mean_shap, index=X_sample.columns).sort_values(ascending=False).head(25)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

colors = ['#ff7f0e' if 'missing' in f else '#2ecc71' if f.startswith('kw_') else
          '#d62728' if any(x in f for x in ['gcs', 'mental', 'shock', 'spo2', 'news2'])
          else '#1f77b4' for f in top_features.index]

axes[0].barh(range(len(top_features)), top_features.values, color=colors)
axes[0].set_yticks(range(len(top_features)))
axes[0].set_yticklabels(top_features.index, fontsize=9)
axes[0].invert_yaxis()
axes[0].set_xlabel('Mean |SHAP Value|', fontsize=11)
axes[0].set_title('Top 25 Features — Global Importance\n(All ESI Classes)', fontsize=12, fontweight='bold')

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#d62728', label='High-signal clinical'),
    Patch(facecolor='#1f77b4', label='Other clinical'),
    Patch(facecolor='#ff7f0e', label='Missingness indicator'),
    Patch(facecolor='#2ecc71', label='Keyword flag'),
]
axes[0].legend(handles=legend_elements)

esi1_shap = pd.Series(np.abs(shap_values[0]).mean(axis=0), index=X_sample.columns).sort_values(ascending=False).head(20)
axes[1].barh(range(len(esi1_shap)), esi1_shap.values, color='#d62728', alpha=0.8)
axes[1].set_yticks(range(len(esi1_shap)))
axes[1].set_yticklabels(esi1_shap.index, fontsize=9)
axes[1].invert_yaxis()
axes[1].set_xlabel('Mean |SHAP Value|', fontsize=11)
axes[1].set_title('Top 20 Features — ESI Level 1 (Critical)\nWhat drives highest-acuity predictions?', fontsize=12, fontweight='bold')

plt.suptitle('Model Explainability via SHAP Values', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('shap_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("Top 10 global features:")
print(top_features.head(10).to_string())

In [ ]:
# conformal prediction gives guaranteed coverage intervals instead of just point predictions
# calibration set (10% holdout) was never used in training — only used here
# q_hat is the nonconformity threshold at 90% coverage
# using the full ensemble probabilities on the calibration set

def build_conformal_sets(cal_probs, cal_labels, query_probs, alpha=0.10):
    n = len(cal_labels)
    nonconf_scores = 1 - cal_probs[np.arange(n), cal_labels - 1]
    q_level = min(np.ceil((n + 1) * (1 - alpha)) / n, 1.0)
    q_hat = np.quantile(nonconf_scores, q_level, method='higher')
    sets = []
    for probs in query_probs:
        included = [i + 1 for i, p in enumerate(probs) if (1 - p) <= q_hat]
        sets.append(included if included else [int(np.argmax(probs)) + 1])
    return sets, q_hat

cal_lgb_probs = sum(m.predict(X_cal, num_iteration=m.best_iteration) for m in lgb_models) / N_FOLDS
cal_xgb_probs = sum(m.predict(xgb.DMatrix(X_cal)).reshape(-1, 5) for m in xgb_models) / N_FOLDS
cal_cat_probs = sum(m.predict_proba(X_cal) for m in cat_models) / N_FOLDS
cal_ensemble_probs = w_lgb * cal_lgb_probs + w_xgb * cal_xgb_probs + w_cat * cal_cat_probs

cal_preds = np.argmax(cal_ensemble_probs, axis=1) + 1
print(f"Calibration QWK: {qwk(y_cal, cal_preds):.4f}")

sets_90, q_hat_90 = build_conformal_sets(cal_ensemble_probs, y_cal, test_ensemble_probs, alpha=0.10)
cal_sets_verify, _ = build_conformal_sets(cal_ensemble_probs, y_cal, cal_ensemble_probs, alpha=0.10)
emp_coverage = np.mean([y_cal[i] in cal_sets_verify[i] for i in range(len(y_cal))])

set_sizes_90 = [len(s) for s in sets_90]
singleton_rate = np.mean([len(s) == 1 for s in sets_90])
critical_confident = sum(1 for s in sets_90 if len(s) == 1 and s[0] in [1, 2])

print(f"\nCONFORMAL PREDICTION (90% coverage target):")
print(f"  q_hat threshold:       {q_hat_90:.4f}")
print(f"  Empirical coverage:    {emp_coverage:.4f}")
print(f"  Mean set size:         {np.mean(set_sizes_90):.3f}")
print(f"  Singleton predictions: {singleton_rate*100:.1f}%")
print(f"  High-acuity singletons (ESI 1-2): {critical_confident:,}")

coverage_by_esi = {}
for esi in range(1, 6):
    mask = np.array([y_cal[i] == esi for i in range(len(y_cal))])
    if mask.sum() > 0:
        covered = sum(y_cal[i] in cal_sets_verify[i] for i in np.where(mask)[0])
        coverage_by_esi[f'ESI {esi}'] = covered / mask.sum()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

size_counts = pd.Series(set_sizes_90).value_counts().sort_index()
axes[0].bar(size_counts.index, size_counts.values, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Prediction Set Size', fontsize=11)
axes[0].set_ylabel('Count', fontsize=11)
axes[0].set_title('Conformal Set Size Distribution\n(90% Coverage)', fontsize=12, fontweight='bold')

axes[1].bar(coverage_by_esi.keys(), coverage_by_esi.values(),
            color=['#d62728', '#ff7f0e', '#ffdd57', '#2ca02c', '#1f77b4'])
axes[1].axhline(0.90, color='black', linestyle='--', linewidth=2, label='90% target')
axes[1].set_ylim(0.75, 1.0)
axes[1].set_ylabel('Empirical Coverage', fontsize=11)
axes[1].set_title('Coverage by ESI Level\n(Should be >= 90%)', fontsize=12, fontweight='bold')
axes[1].legend()

alphas = [0.20, 0.15, 0.10, 0.05, 0.01]
mean_sizes = []
for a in alphas:
    s, _ = build_conformal_sets(cal_ensemble_probs, y_cal, test_ensemble_probs, alpha=a)
    mean_sizes.append(np.mean([len(x) for x in s]))

axes[2].plot([f'{(1-a)*100:.0f}%' for a in alphas], mean_sizes, 'o-', color='steelblue', linewidth=2)
axes[2].set_xlabel('Coverage Level', fontsize=11)
axes[2].set_ylabel('Mean Set Size', fontsize=11)
axes[2].set_title('Coverage vs. Set Size Trade-off\n(Efficiency Curve)', fontsize=12, fontweight='bold')
axes[2].grid(alpha=0.3)

plt.suptitle('Conformal Prediction Analysis — Uncertainty Quantification', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('conformal_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# checking if the model makes more mistakes for certain demographic groups
# undertriage = predicted a less urgent level than the true label
# critical miss = true ESI 1 or 2 predicted as ESI 3 or higher (the dangerous error)
# chi-squared test to check if differences between groups are statistically significant

oof_df = X_main.copy()
oof_df['true'] = y_main
oof_df['pred'] = oof_preds
oof_df['undertriage'] = (oof_df['pred'] > oof_df['true']).astype(int)
oof_df['critical_true'] = (oof_df['true'] <= 2).astype(int)
oof_df['critical_missed'] = ((oof_df['true'] <= 2) & (oof_df['pred'] > 2)).astype(int)

enc = artifacts['encoders']

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
axes = axes.flatten()

bias_targets = [('sex', 'Sex'), ('insurance_type', 'Insurance Type'),
                ('language', 'Language'), ('age_group', 'Age Group')]

for i, (col, label) in enumerate(bias_targets):
    if col not in enc:
        continue
    le = enc[col]
    decoded = le.inverse_transform(oof_df[col].values)

    stats_df = pd.DataFrame({'group': decoded, 'undertriage': oof_df['undertriage'].values,
                              'critical_missed': oof_df['critical_missed'].values,
                              'critical_true': oof_df['critical_true'].values})

    undertriage_rate = stats_df.groupby('group')['undertriage'].mean().sort_values(ascending=False)
    critical_miss = stats_df[stats_df['critical_true'] == 1].groupby('group')['critical_missed'].mean()

    x = np.arange(len(undertriage_rate))
    width = 0.35
    axes[i].bar(x - width/2, undertriage_rate.values * 100, width,
                label='Undertriage rate', color='#d62728', alpha=0.8)
    aligned_critical = critical_miss.reindex(undertriage_rate.index).fillna(0)
    axes[i].bar(x + width/2, aligned_critical.values * 100, width,
                label='Critical miss rate', color='#ff7f0e', alpha=0.8)
    axes[i].axhline(undertriage_rate.mean() * 100, color='gray', linestyle='--',
                    alpha=0.7, label=f'Mean: {undertriage_rate.mean()*100:.1f}%')
    axes[i].set_xticks(x)
    axes[i].set_xticklabels(undertriage_rate.index, rotation=30, ha='right', fontsize=9)
    axes[i].set_ylabel('Rate (%)', fontsize=10)
    axes[i].set_title(f'Undertriage & Critical Miss Rate by {label}', fontsize=11, fontweight='bold')
    axes[i].legend(fontsize=8)

    print(f"\n{label.upper()} UNDERTRIAGE RATES:")
    for grp in undertriage_rate.index:
        n = (decoded == grp).sum()
        ut = undertriage_rate[grp] * 100
        cm_rate = critical_miss.get(grp, 0) * 100
        print(f"  {grp:20s} n={n:6,}  undertriage={ut:.1f}%  critical_miss={cm_rate:.1f}%")

    chi2, p_val = stats.chi2_contingency(
        pd.crosstab(decoded, oof_df['undertriage'].values)
    )[:2]
    print(f"  Chi-squared p={p_val:.4f} ({'SIGNIFICANT' if p_val < 0.05 else 'not significant'})")

plt.suptitle('Systematic Undertriage Audit — Demographic Bias Analysis\n(OOF Predictions vs. True ESI Labels)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('bias_audit.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# looking at whether certain nurses have systematically higher undertriage rates
# only including nurses with at least 30 cases so the rates are meaningful
# flagging anyone above mean + 2 standard deviations as an outlier

if 'triage_nurse_id' in enc:
    le_nurse = enc['triage_nurse_id']
    nurse_decoded = le_nurse.inverse_transform(oof_df['triage_nurse_id'].values)

    nurse_stats = pd.DataFrame({
        'nurse': nurse_decoded,
        'undertriage': oof_df['undertriage'].values,
        'critical_missed': oof_df['critical_missed'].values,
        'n': 1
    }).groupby('nurse').agg(
        n=('n', 'sum'),
        undertriage_rate=('undertriage', 'mean'),
        critical_missed_total=('critical_missed', 'sum')
    )

    nurse_stats = nurse_stats[nurse_stats['n'] >= 30].copy()

    print(f"NURSE-LEVEL VARIABILITY (n>=30 cases):")
    print(f"  Nurses analyzed: {len(nurse_stats)}")
    print(f"  Undertriage rate min: {nurse_stats['undertriage_rate'].min():.3f}  "
          f"max: {nurse_stats['undertriage_rate'].max():.3f}  "
          f"std: {nurse_stats['undertriage_rate'].std():.4f}")

    _, p_anova = stats.f_oneway(*[
        oof_df[nurse_decoded == n]['undertriage'].values
        for n in nurse_stats.index
    ])
    print(f"  ANOVA p-value for between-nurse variance: {p_anova:.6f}")

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    axes[0].hist(nurse_stats['undertriage_rate'], bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    axes[0].axvline(nurse_stats['undertriage_rate'].mean(), color='red', linestyle='--',
                    label=f"Mean: {nurse_stats['undertriage_rate'].mean():.3f}")
    axes[0].axvline(nurse_stats['undertriage_rate'].mean() + 2*nurse_stats['undertriage_rate'].std(),
                    color='orange', linestyle='--', label='+2 SD threshold')
    axes[0].set_xlabel('Undertriage Rate')
    axes[0].set_ylabel('Number of Nurses')
    axes[0].set_title('Distribution of Nurse-Level Undertriage Rates\n(Inter-Rater Variability)', fontweight='bold')
    axes[0].legend()

    top20 = nurse_stats.nlargest(20, 'undertriage_rate')
    threshold = nurse_stats['undertriage_rate'].mean() + 2 * nurse_stats['undertriage_rate'].std()
    bar_colors = ['#d62728' if r > threshold else '#1f77b4' for r in top20['undertriage_rate'].values]
    axes[1].barh(range(len(top20)), top20['undertriage_rate'].values, color=bar_colors)
    axes[1].axvline(threshold, color='orange', linestyle='--', label=f'+2 SD: {threshold:.3f}')
    axes[1].set_yticks(range(len(top20)))
    axes[1].set_yticklabels([f"{n} (n={int(top20.iloc[i]['n'])})" for i, n in enumerate(top20.index)], fontsize=8)
    axes[1].invert_yaxis()
    axes[1].set_xlabel('Undertriage Rate')
    axes[1].set_title('Top 20 Nurses by Undertriage Rate\n(Red = >2 SD above mean)', fontweight='bold')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig('nurse_variability.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# final predictions from the weighted ensemble of all three models
# saving to submission.csv in the required format

test_preds = np.argmax(test_ensemble_probs, axis=1) + 1

submission = pd.read_csv(PATH + 'sample_submission.csv')
submission['triage_acuity'] = test_preds
submission.to_csv('submission.csv', index=False)

cm = confusion_matrix(y_main, oof_preds)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=[f'ESI {i}' for i in range(1, 6)],
            yticklabels=[f'ESI {i}' for i in range(1, 6)])
axes[0].set_xlabel('Predicted', fontsize=11)
axes[0].set_ylabel('True', fontsize=11)
axes[0].set_title(f'Ensemble OOF Confusion Matrix\nQWK: {oof_qwk:.4f}', fontsize=12, fontweight='bold')

categories = ['QWK', 'Accuracy']
our_scores = [oof_qwk, oof_acc]
news2_scores = [0.7723, 0.4076]

x = np.arange(len(categories))
axes[1].bar(x - 0.2, our_scores, 0.35, label='Our Ensemble (LGB+XGB+CAT)', color='steelblue')
axes[1].bar(x + 0.2, news2_scores, 0.35, label='NEWS2 Clinical Baseline', color='#ff7f0e')
axes[1].set_xticks(x)
axes[1].set_xticklabels(categories, fontsize=12)
axes[1].set_ylim(0, 1.0)
axes[1].set_title('Ensemble vs. NEWS2 Clinical Baseline', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=11)
for j, (our, base) in enumerate(zip(our_scores, news2_scores)):
    axes[1].text(j - 0.2, our + 0.01, f'{our:.4f}', ha='center', fontsize=10, fontweight='bold')
    axes[1].text(j + 0.2, base + 0.01, f'{base:.4f}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('final_results.png', dpi=150, bbox_inches='tight')
plt.show()

print("FINAL SUBMISSION SUMMARY")
print(f"  LightGBM OOF QWK:  {lgb_oof_qwk:.4f}")
print(f"  XGBoost OOF QWK:   {xgb_oof_qwk:.4f}")
print(f"  CatBoost OOF QWK:  {cat_oof_qwk:.4f}")
print(f"  Ensemble OOF QWK:  {oof_qwk:.4f}")
print(f"  Ensemble Accuracy: {oof_acc:.4f}")
print(f"  NEWS2 QWK:         0.7723")
print(f"  QWK improvement:   {oof_qwk - 0.7723:+.4f}")
print(f"\nEnsemble weights: LGB={w_lgb:.3f} | XGB={w_xgb:.3f} | CAT={w_cat:.3f}")
print(f"\nPredicted test distribution:")
for i in range(1, 6):
    n = (test_preds == i).sum()
    print(f"  ESI {i}: {n:,} ({n/len(test_preds)*100:.1f}%)")
print(f"\nFiles saved: submission.csv")

In [ ]:
print("LEAKAGE AUDIT")
print("Columns excluded from modeling (post-triage information):")
print("  - disposition: outcome known only after triage decision")
print("  - ed_los_hours: ED length of stay known only at discharge")
print("  - triage_acuity: target variable")
print("\nColumns confirmed safe (known at triage time):")
safe = ['news2_score', 'gcs_total', 'mental_status_triage', 
        'triage_nurse_id', 'site_id', 'arrival_mode']
for c in safe:
    print(f"  - {c}")
print("\nFeature matrix shape going into model:", X_full.shape)
print("No test set information used during training or calibration.")